<a href="https://colab.research.google.com/github/nadirju/AQM_APP/blob/main/AQM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import logging
import requests
import gradio as gr
from openai import OpenAI
from google.colab import userdata
userdata.get('AQMKey')

# ==========================================
# LOGGING CONFIGURATION
# ==========================================
# Set up logging to print detailed stack traces to the console/terminal
# while keeping the Gradio UI clean and user-friendly.
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# ==========================================
# API CLIENT INITIALIZATION
# ==========================================
# Security Note: In a real production environment, never hardcode API keys.
# Use environment variables or Gradio Secrets.
XAI_API_KEY = os.environ.get("XAI_API_KEY", "AQMKey")

try:
    client = OpenAI(
        api_key=XAI_API_KEY,
        base_url="https://api.x.ai/v1"
    )
except Exception as e:
    logger.error(f"Failed to initialize OpenAI client: {e}", exc_info=True)
    client = None

# ==========================================
# HELPER FUNCTIONS
# ==========================================

def classify_aqi(aqi: int) -> str:
    """Classify US AQI into a human-readable category with emoji."""
    if aqi <= 50:
        return "Good 🟢"
    elif aqi <= 100:
        return "Moderate 🟡"
    elif aqi <= 150:
        return "Unhealthy for Sensitive Groups 🟠"
    elif aqi <= 200:
        return "Unhealthy 🔴"
    elif aqi <= 300:
        return "Very Unhealthy 🟣"
    else:
        return "Hazardous ⚫"


def get_coordinates(location: str) -> dict | None:
    """
    Fetch coordinates for a given location using Open-Meteo Geocoding API.
    Returns a dictionary of location data or None if failed.
    """
    if not location or not isinstance(location, str) or not location.strip():
        return None

    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {
        "name": location.strip(),
        "count": 1,
        "language": "en",
        "format": "json"
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()  # Raises HTTPError for 4xx/5xx responses
        data = response.json()

        # API Data Verification: Safely check expected fields
        if "results" not in data or not data["results"]:
            logger.warning(f"No geocoding results found for: {location}")
            return None

        result = data["results"][0]

        # Safely extract data using .get() to prevent KeyError
        return {
            "name": result.get("name", location),
            "country": result.get("country", "Unknown"),
            "latitude": result.get("latitude"),
            "longitude": result.get("longitude")
        }

    except requests.exceptions.RequestException as e:
        logger.error(f"Network/API error fetching coordinates for '{location}': {e}", exc_info=True)
        return None
    except ValueError as e:
        logger.error(f"JSON decode error for coordinates of '{location}': {e}", exc_info=True)
        return None
    except Exception as e:
        logger.error(f"Unexpected error fetching coordinates for '{location}': {e}", exc_info=True)
        return None


def get_air_quality(latitude: float, longitude: float) -> dict:
    """
    Fetch current air quality data from Open-Meteo.
    Returns a dictionary containing the data or an 'error' key.
    """
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": [
            "us_aqi", "european_aqi", "pm2_5", "pm10",
            "carbon_monoxide", "nitrogen_dioxide", "sulphur_dioxide", "ozone"
        ],
        "timezone": "auto"
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        return response.json()

    except requests.exceptions.RequestException as e:
        logger.error(f"Network/API error fetching air quality for ({latitude}, {longitude}): {e}", exc_info=True)
        return {"error": "Network or API error while fetching air quality data."}
    except ValueError as e:
        logger.error(f"JSON decode error for air quality: {e}", exc_info=True)
        return {"error": "Invalid response format from air quality API."}
    except Exception as e:
        logger.error(f"Unexpected error fetching air quality: {e}", exc_info=True)
        return {"error": "An unexpected error occurred while fetching air quality data."}


def get_ai_advice(location: str, air_quality: dict) -> str:
    """
    Generate AI advisory using the xAI API.
    Returns a string with the advisory or a user-friendly error message.
    """
    if client is None:
        return "AI advisory is currently unavailable because the API client failed to initialize."

    current = air_quality.get("current", {})

    prompt = f"""
You are an air quality and smog advisory assistant.

Analyze the following air quality data for:

Location: {location}

US AQI: {current.get("us_aqi")}
European AQI: {current.get("european_aqi")}
PM2.5: {current.get("pm2_5")} µg/m³
PM10: {current.get("pm10")} µg/m³
Carbon Monoxide: {current.get("carbon_monoxide")} µg/m³
Nitrogen Dioxide: {current.get("nitrogen_dioxide")} µg/m³
Sulphur Dioxide: {current.get("sulphur_dioxide")} µg/m³
Ozone: {current.get("ozone")} µg/m³

Provide a clear and practical air quality advisory.

Include:
1. Overall air quality assessment.
2. Whether smog or particulate pollution appears concerning.
3. Advice for the general public.
4. Advice for children and elderly people.
5. Advice for people with asthma or respiratory conditions.
6. Whether outdoor exercise is recommended.
7. Precautions that should be taken.

Use simple language.

Important:
Do not diagnose medical conditions.
Clearly state that the advice is general environmental information.
"""

    try:
        response = client.responses.create(
            model="openai/gpt-oss-120b",
            input=prompt
        )
        return response.output_text
    except Exception as e:
        # Catch any AI API errors (timeout, auth, rate limit) and log the stack trace
        logger.error(f"AI Advisory API error: {e}", exc_info=True)
        return "AI advisory is currently unavailable due to a service error. Please try again later."


def search_cities(search_text: str) -> dict:
    """
    Search for cities using Open-Meteo Geocoding API and return a Gradio dropdown update.
    """
    if not search_text or not isinstance(search_text, str) or not search_text.strip():
        return gr.update(choices=[], value=None)

    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {
        "name": search_text.strip(),
        "count": 5,
        "language": "en",
        "format": "json"
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        if "results" not in data or not data["results"]:
            return gr.update(choices=[], value=None)

        cities = []
        for result in data["results"]:
            city = result.get("name", "")
            country = result.get("country", "")
            admin1 = result.get("admin1", "")

            if admin1:
                display_name = f"{city}, {admin1}, {country}"
            else:
                display_name = f"{city}, {country}"

            if display_name.strip():
                cities.append(display_name)

        selected_value = cities[0] if cities else None
        return gr.update(choices=cities, value=selected_value)

    except requests.exceptions.RequestException as e:
        logger.error(f"Network/API error during city search: {e}", exc_info=True)
        return gr.update(choices=[], value=None)
    except Exception as e:
        logger.error(f"Unexpected error during city search: {e}", exc_info=True)
        return gr.update(choices=[], value=None)


def analyze_location(location: str) -> tuple[str, str]:
    """
    Main backend function triggered by the Gradio button.
    Returns a tuple of exactly two strings: (Air Quality Summary, AI Advisory).
    """
    # 1. Input Validation
    if not location or not isinstance(location, str) or not location.strip():
        return (
            "### Error: Please enter or select a valid location.",
            "AI advisory not available due to invalid location input."
        )

    # 2. Get Coordinates
    coordinates = get_coordinates(location)
    if coordinates is None:
        return (
            f"### Error: Location '{location}' not found.",
            "AI advisory not available because location could not be determined. Check your spelling or try a more specific location."
        )

    # 3. Get Air Quality
    air_quality = get_air_quality(coordinates["latitude"], coordinates["longitude"])

    if "error" in air_quality:
        return (
            f"### Error: Failed to retrieve air quality data for {location}.\nDetails: {air_quality['error']}",
            "AI advisory not available due to an error fetching air quality data. Please try again later."
        )

    # 4. Parse Air Quality Data Safely
    current = air_quality.get("current", {})
    us_aqi = current.get("us_aqi")

    aqi_status = classify_aqi(us_aqi) if us_aqi is not None else "Not available"

    # 5. Build Summary Markdown
    summary = f"""
### 📍 Location

**City:** {coordinates.get('name', 'N/A')}

**Country:** {coordinates.get('country', 'N/A')}

---

### 🌫️ Air Quality Status

**US AQI:** {us_aqi if us_aqi is not None else 'N/A'}

**Status:** {aqi_status}

---

### 🧪 Pollutants

| Pollutant | Value |
|---|---:|
| PM2.5 | {current.get('pm2_5', 'N/A')} µg/m³ |
| PM10 | {current.get('pm10', 'N/A')} µg/m³ |
| Carbon Monoxide | {current.get('carbon_monoxide', 'N/A')} µg/m³ |
| Nitrogen Dioxide | {current.get('nitrogen_dioxide', 'N/A')} µg/m³ |
| Sulphur Dioxide | {current.get('sulphur_dioxide', 'N/A')} µg/m³ |
| Ozone | {current.get('ozone', 'N/A')} µg/m³ |
"""

    # 6. Get AI Advisory (Now wrapped safely)
    ai_advice = get_ai_advice(
        f"{coordinates.get('name', 'N/A')}, {coordinates.get('country', 'N/A')}",
        air_quality
    )

    # 7. Return exactly 2 values to match the 2 Gradio output components
    return summary, ai_advice


# ==========================================
# GRADIO INTERFACE
# ==========================================

with gr.Blocks(title="SMOG AND AIR QUALITY ADVISORY") as demo:

    gr.Markdown("""
    # 🌍 SMOG AND AIR QUALITY ADVISORY

    ### Check air pollution, smog conditions and receive an AI-powered advisory.
    """)

    # Location Search
    location_search = gr.Textbox(
        label="📍 Search Your City",
        placeholder="Type a city name, e.g. Islamabad"
    )

    city_dropdown = gr.Dropdown(
        choices=[],
        label="Select Your City",
        interactive=True
    )

    # Button
    analyze_button = gr.Button(
        "🌫️ Check Air Quality",
        variant="primary"
    )

    # Outputs
    with gr.Row():
        with gr.Column():
            air_quality_output = gr.Markdown("Air quality information will appear here.")
        with gr.Column():
            ai_advice_output = gr.Markdown("AI-generated advisory will appear here.")

    # Live City Search Event
    location_search.input(
        fn=search_cities,
        inputs=location_search,
        outputs=city_dropdown
    )

    # Analyze Button Event
    # Note: The outputs list contains exactly 2 components.
    # analyze_location returns exactly 2 strings. This alignment prevents the Gradio 'Error' state.
    analyze_button.click(
        fn=analyze_location,
        inputs=city_dropdown,
        outputs=[
            air_quality_output,
            ai_advice_output
        ]
    )

if __name__ == "__main__":
    demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d601e588d7d8a611b3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


KeyboardInterrupt: 